# powregister CUDA GPU Test

**Setup:** Runtime > Change runtime type > **T4 GPU**

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Install dependencies
import sys

py_ver = f"{sys.version_info.major}.{sys.version_info.minor}"
print(f"Python version: {py_ver}")

!pip install pycryptodome 2>&1 | tail -1

# cubit: OpenTensor CUDA solver (not the PyPI 'cubit' package)
# Pre-built wheels available for Python 3.7-3.10 only
if sys.version_info.minor <= 10:
    tag = f"cp3{sys.version_info.minor}-cp3{sys.version_info.minor}"
    url = f"https://github.com/opentensor/cubit/releases/download/v1.1.2/cubit-1.1.2-{tag}-linux_x86_64.whl"
    print(f"Installing cubit from: {url}")
    !pip install {url} 2>&1 | tail -1
else:
    print(f"WARNING: No pre-built cubit wheel for Python {py_ver}")
    print("Building from source...")
    !pip install git+https://github.com/opentensor/cubit.git 2>&1 | tail -3

print("\nInstallation complete.")

In [ ]:
# PoW core functions (from core.py)
import binascii
import hashlib
import multiprocessing as mp
import random
import time
from dataclasses import dataclass
from typing import List, Optional, Tuple, Union

from Crypto.Hash import keccak

CUDA_AVAILABLE = False
try:
    import cubit

    CUDA_AVAILABLE = True
except ImportError:
    cubit = None

try:
    import torch

    TORCH_CUDA_AVAILABLE = torch.cuda.is_available()
except ImportError:
    TORCH_CUDA_AVAILABLE = False


@dataclass
class POWSolution:
    nonce: int
    block_number: int
    difficulty: int
    seal: bytes


def hex_bytes_to_u8_list(hex_bytes: bytes) -> list:
    return [int(hex_bytes[i : i + 2], 16) for i in range(0, len(hex_bytes), 2)]


def hash_block_with_hotkey(block_bytes: bytes, hotkey_bytes: bytes) -> bytes:
    kec = keccak.new(digest_bits=256)
    kec.update(bytearray(block_bytes + hotkey_bytes))
    return kec.digest()


def create_seal_hash(block_and_hotkey_hash: bytes, nonce: int) -> bytes:
    nonce_bytes = binascii.hexlify(nonce.to_bytes(8, "little"))
    hash_hex = binascii.hexlify(block_and_hotkey_hash)[:64]
    pre_seal = nonce_bytes + hash_hex
    sha256_result = hashlib.sha256(bytearray(hex_bytes_to_u8_list(pre_seal))).digest()
    kec = keccak.new(digest_bits=256)
    seal = kec.update(sha256_result).digest()
    return seal


def seal_meets_difficulty(seal: bytes, difficulty: int) -> bool:
    seal_number = int.from_bytes(seal, "little")
    product = seal_number * difficulty
    return product < (2**256)


def _format_hashrate(h: float) -> str:
    if h >= 1_000_000:
        return f"{h / 1_000_000:.2f} MH/s"
    elif h >= 1_000:
        return f"{h / 1_000:.2f} KH/s"
    return f"{h:.0f} H/s"


def solve_cuda(
    nonce_start: int,
    update_interval: int,
    tpb: int,
    block_and_hotkey_hash: bytes,
    difficulty: int,
    dev_id: int = 0,
) -> Tuple[int, Optional[bytes]]:
    if not CUDA_AVAILABLE:
        raise ImportError("cubit not installed. Run: pip install cubit")
    limit = 2**256 - 1
    upper = limit // difficulty
    upper_bytes = upper.to_bytes(32, byteorder="little", signed=False)
    block_and_hotkey_hash_hex = binascii.hexlify(block_and_hotkey_hash)[:64]
    solution_nonce = cubit.solve_cuda(
        tpb,
        nonce_start,
        update_interval,
        upper_bytes,
        block_and_hotkey_hash_hex,
        dev_id,
    )
    if solution_nonce != -1:
        seal = create_seal_hash(block_and_hotkey_hash, solution_nonce)
        if seal_meets_difficulty(seal, difficulty):
            return solution_nonce, seal
    return -1, None


def _cuda_worker_solve(
    block_and_hotkey_hash,
    difficulty,
    block_number,
    nonce_start,
    update_interval,
    tpb,
    dev_id,
    result_queue,
    stop_event,
    progress_counter=None,
):
    nonce_limit = 2**64 - 1
    current_nonce = nonce_start
    nonces_per_batch = update_interval * tpb
    while not stop_event.is_set():
        try:
            solution_nonce, seal = solve_cuda(
                nonce_start=current_nonce,
                update_interval=update_interval,
                tpb=tpb,
                block_and_hotkey_hash=block_and_hotkey_hash,
                difficulty=difficulty,
                dev_id=dev_id,
            )
            if solution_nonce != -1 and seal is not None:
                result_queue.put(
                    POWSolution(
                        nonce=solution_nonce,
                        block_number=block_number,
                        difficulty=difficulty,
                        seal=seal,
                    )
                )
                return
            if progress_counter is not None:
                with progress_counter.get_lock():
                    progress_counter.value += nonces_per_batch
            current_nonce = (current_nonce + nonces_per_batch) % nonce_limit
        except Exception as e:
            print(f"CUDA worker error: {e}")
            return


def validate_tpb(tpb: int) -> int:
    if tpb % 32 != 0:
        new_tpb = ((tpb // 32) + 1) * 32
        print(f"Warning: tpb should be multiple of 32. Adjusting {tpb} -> {new_tpb}")
        return new_tpb
    return tpb


def solve_pow_cuda(
    block_and_hotkey_hash: bytes,
    difficulty: int,
    block_number: int,
    dev_id: Union[int, List[int]] = 0,
    tpb: int = 256,
    update_interval: int = 100_000,
    timeout: int = 3600,
) -> Optional[POWSolution]:
    if not CUDA_AVAILABLE:
        raise ImportError("cubit not installed. Run: pip install cubit")
    tpb = validate_tpb(tpb)
    dev_ids = [dev_id] if isinstance(dev_id, int) else list(dev_id)
    result_queue = mp.Queue()
    stop_event = mp.Event()
    progress_counter = mp.Value("L", 0)
    processes = []
    nonce_limit = 2**64 - 1
    for device in dev_ids:
        nonce_start = random.randint(0, nonce_limit)
        p = mp.Process(
            target=_cuda_worker_solve,
            args=(
                block_and_hotkey_hash,
                difficulty,
                block_number,
                nonce_start,
                update_interval,
                tpb,
                device,
                result_queue,
                stop_event,
                progress_counter,
            ),
            daemon=True,
        )
        processes.append(p)
        p.start()
    start_time = time.time()
    try:
        while True:
            elapsed = time.time() - start_time
            if elapsed >= timeout:
                print(f"\nCUDA timeout after {elapsed:.0f}s")
                stop_event.set()
                return None
            try:
                solution = result_queue.get(timeout=3)
                stop_event.set()
                total_hashes = progress_counter.value
                print(
                    f"CUDA solved! {total_hashes:,} hashes in {elapsed:.1f}s "
                    f"({_format_hashrate(total_hashes / max(elapsed, 0.001))})"
                )
                return solution
            except Exception:
                total_hashes = progress_counter.value
                if total_hashes > 0:
                    rate = total_hashes / max(elapsed, 0.001)
                    print(
                        f"\r  [{elapsed:.0f}s] CUDA {total_hashes:,} hashes | "
                        f"{_format_hashrate(rate)} | "
                        f"difficulty: {difficulty:,}",
                        end="",
                        flush=True,
                    )
    finally:
        stop_event.set()
        for p in processes:
            p.terminate()


print("Functions loaded.")

## CUDA Status

In [ ]:
print("=== CUDA Status ===")
print(f"cubit available: {CUDA_AVAILABLE}")
print(f"torch CUDA available: {TORCH_CUDA_AVAILABLE}")
if TORCH_CUDA_AVAILABLE:
    props = torch.cuda.get_device_properties(0)
    print(f"GPU count: {torch.cuda.device_count()}")
    print(f"GPU name: {props.name}")
    print(f"VRAM total: {props.total_memory / 1024**3:.1f} GB")
    print(f"VRAM free: {torch.cuda.mem_get_info(0)[0] / 1024**3:.1f} GB")
    print(f"CUDA cores (SM x cores): {props.multi_processor_count} SMs")
    print(f"Compute capability: {props.major}.{props.minor}")
print(f"Recommended solver: {'CUDA' if CUDA_AVAILABLE else 'CPU'}")

## Test 1: Hash Computation

In [ ]:
print("=== Hash Computation Test ===")
test_block = bytes(32)
test_hotkey = bytes(32)
block_and_hotkey = hash_block_with_hotkey(test_block, test_hotkey)
print(f"Block+Hotkey Hash: {block_and_hotkey.hex()[:32]}...")

seal = create_seal_hash(block_and_hotkey, 0)
print(f"Seal Hash: {seal.hex()[:32]}...")
print(f"Meets difficulty 10M: {seal_meets_difficulty(seal, 10_000_000)}")

## Test 2: Single CUDA Solve (low difficulty)

In [ ]:
if not CUDA_AVAILABLE:
    print("CUDA not available! Check GPU runtime.")
    print("Runtime > Change runtime type > T4 GPU")
else:
    print("=== Single CUDA Solve (difficulty: 1,000) ===")
    start = time.time()
    nonce, seal = solve_cuda(
        nonce_start=0,
        update_interval=100_000,
        tpb=256,
        block_and_hotkey_hash=block_and_hotkey,
        difficulty=1000,
        dev_id=0,
    )
    elapsed = time.time() - start
    if nonce != -1:
        print(f"Solution found! nonce={nonce}, time={elapsed:.3f}s")
        print(f"Seal: {seal.hex()[:32]}...")
    else:
        print(f"No solution found ({elapsed:.3f}s)")

## Test 3: Parallel CUDA Solve (medium difficulty)

In [ ]:
if not CUDA_AVAILABLE:
    print("CUDA not available!")
else:
    print("=== Parallel CUDA Solve (difficulty: 10,000) ===")
    start = time.time()
    solution = solve_pow_cuda(
        block_and_hotkey_hash=block_and_hotkey,
        difficulty=10_000,
        block_number=1000,
        dev_id=0,
        tpb=256,
        update_interval=100_000,
        timeout=60,
    )
    elapsed = time.time() - start
    if solution:
        print(f"\nSolution found! nonce={solution.nonce}, time={elapsed:.3f}s")
    else:
        print(f"\nNo solution found ({elapsed:.1f}s)")

## Test 4: High Difficulty (500M)

In [ ]:
if not CUDA_AVAILABLE:
    print("CUDA not available!")
else:
    print("=== High Difficulty Test (500,000,000) ===")
    start = time.time()
    solution = solve_pow_cuda(
        block_and_hotkey_hash=block_and_hotkey,
        difficulty=500_000_000,
        block_number=1000,
        dev_id=0,
        tpb=256,
        update_interval=100_000,
        timeout=300,
    )
    elapsed = time.time() - start
    if solution:
        print(f"\n500M difficulty solved! nonce={solution.nonce}, time={elapsed:.1f}s")
    else:
        print(f"\nNo solution found ({elapsed:.1f}s)")